# 01 – Data Source Layer: Calling the APIs

**Free Power Hours** – Big Data Analytics final project

This notebook shows **how we get data from our two phase-1 sources**:

| # | Source | What | Access |
|---|---|---|---|
| 1 | [aWATTar API](https://www.awattar.at/services/api) | Austrian day-ahead electricity prices (EUR/MWh) | REST/JSON, free, no key, fair use ~100 req/day |
| 2 | [Open-Meteo](https://open-meteo.com/en/docs) | Hourly weather for 10 AT/DE locations | REST/JSON, free, no key, CC BY 4.0 |

Structure:
1. **Part A** – a raw API call "by hand" with `requests` (aWATTar)
2. **Part B** – a raw API call to Open-Meteo
3. **Part C** – the same with our reusable project module `src/ingest` (batching + caching)
4. **Part D** – a first look: do sunny hours mean cheap electricity?

> ENTSO-E (generation & load) follows in phase 2.

## Setup

In [ ]:
import sys, json
from pathlib import Path

import requests
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Make the project package `src` importable from inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Python    ", sys.version.split()[0])
for lib in (requests, pd, matplotlib):
    print(f"{lib.__name__:<10}", lib.__version__)

## Part A – Calling the aWATTar API by hand

An API call always has the same ingredients:

1. **Endpoint URL** – `https://api.awattar.at/v1/marketdata`
2. **Parameters** – here `start` and `end` as **Unix epoch milliseconds** (ms since 1970-01-01 UTC)
3. **HTTP request** – `GET`
4. **Response** – status code (200 = OK) + body (JSON)

Let's ask for the prices of the last 7 days.

In [ ]:
end = pd.Timestamp.now(tz="UTC").floor("D")          # today 00:00 UTC
start = end - pd.Timedelta(days=7)

params = {
    "start": int(start.timestamp() * 1000),          # epoch milliseconds
    "end":   int(end.timestamp() * 1000),
}
response = requests.get("https://api.awattar.at/v1/marketdata", params=params, timeout=30)

print("URL:        ", response.url)
print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

The body is JSON. `response.json()` turns it into Python dicts/lists. Let's look at the structure:

In [ ]:
data = response.json()
print("Top-level keys:", list(data.keys()))
print("Number of records:", len(data["data"]))
print(json.dumps(data["data"][:2], indent=2))

Each record is one market interval:

| field | meaning |
|---|---|
| `start_timestamp`, `end_timestamp` | interval in epoch **ms**, UTC |
| `marketprice` | price in EUR/MWh (can be **negative**) |
| `unit` | always `Eur/MWh` |

Now convert it to a pandas DataFrame with real timestamps. We **keep everything in UTC** internally
and only convert to Vienna time for display (avoids daylight-saving-time problems).

In [ ]:
prices = pd.DataFrame(data["data"])
prices["ts"] = pd.to_datetime(prices["start_timestamp"], unit="ms", utc=True)
prices["ts_vienna"] = prices["ts"].dt.tz_convert("Europe/Vienna")
prices = prices.rename(columns={"marketprice": "price_eur_mwh"})[["ts", "ts_vienna", "price_eur_mwh"]]
prices.head()

In [ ]:
ax = prices.plot(x="ts_vienna", y="price_eur_mwh", figsize=(12, 4), legend=False,
                 title="aWATTar day-ahead price – last 7 days")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Vienna local time"); ax.set_ylabel("EUR/MWh");

## Part B – Calling Open-Meteo by hand

Open-Meteo has **three endpoints** that we use for different purposes:

| endpoint | content | used for |
|---|---|---|
| `archive-api.open-meteo.com/v1/archive` | actual past weather (ERA5 reanalysis) | exploratory analysis |
| `historical-forecast-api.open-meteo.com/v1/forecast` | what the weather models *forecast* back then (≈2022+) | **ML training** – our model must work with forecasts, not with perfect hindsight |
| `api.open-meteo.com/v1/forecast` | live forecast | "tomorrow's cheap hours" demo |

Parameters: coordinates, the list of `hourly` variables, a date range and `timezone=UTC`.

In [ ]:
params = {
    "latitude": 48.21, "longitude": 16.37,           # Vienna
    "hourly": "temperature_2m,shortwave_radiation,cloud_cover,wind_speed_100m",
    "start_date": start.strftime("%Y-%m-%d"),
    "end_date": (end - pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    "timezone": "UTC",
}
# The archive lags a few days behind -> for the last week we use the historical-forecast endpoint
r = requests.get("https://historical-forecast-api.open-meteo.com/v1/forecast", params=params, timeout=60)
print(r.status_code, r.url[:120], "...")
weather_raw = r.json()
print("Top-level keys:", list(weather_raw.keys()))
print("Units:", weather_raw["hourly_units"])

The `hourly` block is **column-oriented**: one list per variable, all the same length.
That maps directly to a DataFrame:

In [ ]:
weather = pd.DataFrame(weather_raw["hourly"])
weather["ts"] = pd.to_datetime(weather.pop("time"), utc=True)
weather.head()

## Part C – The same with our project module `src/ingest`

Calling APIs by hand is fine for exploring, but for **7 years × 10 locations** we need:

* **batching** – aWATTar: one request per year (the API returns 8,760 hours at once), Open-Meteo: one request per year with all 10 locations
* **caching** – each raw response is stored once in `data/raw/…` (the *raw zone*); re-running never hits the API again
* **metadata** – every cached file records `source`, `fetched_at` and the request `params` (data lineage)
* **retries** – on network errors / HTTP 429 / 5xx

All endpoints, locations and variables are defined in one place: `src/ingest/config.py`.

In [ ]:
from src.ingest import config, awattar, openmeteo

pd.DataFrame(config.LOCATIONS, index=["country", "lat", "lon", "why"]).T

In [ ]:
# May + June 2024: a sunny period with many negative prices
# Fetches/caches whole years (one request per year), then filters to the range
prices_2024 = awattar.load_prices("2024-05-01", "2024-07-01")
print(prices_2024.shape)
prices_2024.head()

A cached raw file looks like this – the unmodified API response plus our metadata:

In [ ]:
raw = json.loads((config.RAW_DIR / "awattar" / "2024.json").read_text())
{k: (v if k != "payload" else f"<{len(v['data'])} records>") for k, v in raw.items()}

In [ ]:
# All 10 locations for 2024 in one request (actual weather from the archive)
weather_2024 = openmeteo.to_dataframe(openmeteo.fetch_year("archive", 2024))
print(weather_2024.shape, "->", weather_2024["location"].nunique(), "locations")
weather_2024.head()

## Part D – First look: sun vs. price

Join the price with the solar radiation in Vienna and Munich (south-German solar) on the UTC timestamp.

In [ ]:
solar = (weather_2024[weather_2024["location"].isin(["vienna", "munich"])]
         .pivot(index="ts", columns="location", values="shortwave_radiation")
         .add_prefix("radiation_"))
df = prices_2024.set_index("ts")[["price_eur_mwh"]].join(solar, how="inner")

week = df.loc["2024-05-10":"2024-05-17"]
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(week.index, week["price_eur_mwh"], color="tab:blue", label="price")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_ylabel("EUR/MWh", color="tab:blue")
ax2 = ax1.twinx()
ax2.fill_between(week.index, week["radiation_munich"], color="orange", alpha=0.3, label="radiation Munich")
ax2.set_ylabel("W/m²", color="darkorange")
ax1.set_title("Midday sun pushes the price down – 10–17 May 2024");

In [ ]:
neg = df["price_eur_mwh"] < 0
print(f"Hours in May+June 2024:  {len(df)}")
print(f"Negative-price hours:    {neg.sum()} ({neg.mean():.1%})")
print(f"Mean radiation Munich when price < 0:  {df.loc[neg, 'radiation_munich'].mean():.0f} W/m²")
print(f"Mean radiation Munich when price >= 0: {df.loc[~neg, 'radiation_munich'].mean():.0f} W/m²")

## Next steps (Data Source Layer)

- [x] Fetch full history into the raw cache (aWATTar 2019–2026: 8 requests; Open-Meteo archive 2019–2026, historical forecast 2022–2026)
- [ ] Publish to Kafka: `python -m src.ingest.stream backfill ...` (see README)
- [ ] Live feed for the demo: `python -m src.ingest.stream live --interval 900`
- [ ] Data quality checks: missing hours, DST duplicates/gaps, outliers, coverage per source
- [ ] Phase 2: ENTSO-E generation/load, holidays
- [ ] Hand-over: Kafka topics are consumed by Spark Structured Streaming (storage/processing layer)

Reference documentation: [data sources](../docs/data-sources.md), [Kafka data contract](../docs/data-contract.md), [architecture](../docs/architecture.md).